# Train Your MnistNet through Backpropagation

## MnistNet class

Let's import Layer and Utility classes

In [ ]:
from layer import Layer
from utility import Utility


MnistNet class and utility functions for training and evaluation of the model.

Network Architecture: (three-layer fully connected neural network)
* Input size: 784 (28x28 images)
* 1st Hidden layer size: 100
* 2nd Hidden layer size: 50 
* Output size: 10 (digits 0-9)

In [ ]:
from collections import OrderedDict
import numpy as np

INPUT_SIZE = 784
OUTPUT_SIZE = 10

class MnistNet:
    def __init__(self, hidden1_size, hidden2_size):
        self.params = {}

        w1_scale = np.sqrt(2.0 / INPUT_SIZE)
        w2_scale = np.sqrt(2.0 / hidden1_size)
        w3_scale = np.sqrt(2.0 / hidden2_size)

        self.params['w1'] = w1_scale * np.random.randn(INPUT_SIZE, hidden1_size)
        self.params['b1'] = np.zeros(hidden1_size)
        self.params['w2'] = w2_scale * np.random.randn(hidden1_size, hidden2_size)
        self.params['b2'] = np.zeros(hidden2_size)
        self.params['w3'] = w3_scale * np.random.randn(hidden2_size, OUTPUT_SIZE)
        self.params['b3'] = np.zeros(OUTPUT_SIZE)

        self.layers = OrderedDict()
        self.layers['Affine1'] = Layer.Affine(self.params['w1'], self.params['b1'])
        self.layers['Relu1'] = Layer.Relu()
        self.layers['Affine2'] = Layer.Affine(self.params['w2'], self.params['b2'])
        self.layers['Relu2'] = Layer.Relu()
        self.layers['Affine3'] = Layer.Affine(self.params['w3'], self.params['b3'])
        self.last_layer = Layer.SoftmaxWithLoss()

    def predict(self, x):
        for layer in self.layers.values():
            x = layer.forward(x)
        return x

    def loss(self, x, y):
        y_hat = self.predict(x)
        return self.last_layer.forward(y_hat, y)

    def accuracy(self, x, y):
        y_hat = self.predict(x)
        y_pred = np.argmax(y_hat, axis=1)
        y_true = np.argmax(y, axis=1) if y.ndim == 2 else y
        return np.mean(y_pred == y_true)

    def gradient(self, x, y):
        self.loss(x, y)

        dout = self.last_layer.backward(1)
        layers = list(self.layers.values())
        layers.reverse()

        for layer in layers:
            dout = layer.backward(dout)

        grads = {}
        grads['w1'] = self.layers['Affine1'].dw
        grads['b1'] = self.layers['Affine1'].db
        grads['w2'] = self.layers['Affine2'].dw
        grads['b2'] = self.layers['Affine2'].db
        grads['w3'] = self.layers['Affine3'].dw
        grads['b3'] = self.layers['Affine3'].db
        return grads


In [ ]:
HIDDEN1_SIZE = 100
HIDDEN2_SIZE = 50

np.random.seed(1)
net = MnistNet(HIDDEN1_SIZE, HIDDEN2_SIZE)


## Load Training/Test Dataset Using MnistData

Use your `MnistData` class to load `x_train, y_train` and `x_test, y_test`.

In [ ]:
from mnist_data import MnistData

mnist = MnistData()


In [ ]:
x_train, y_train, x_test, y_test = mnist.load_data()
print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)


## Hyperparameters:

In [ ]:
epoch_num = 3
batch_size = 100
train_size = x_train.shape[0]
learning_rate = 0.1


## Training

In [ ]:
import numpy as np

def train(net, learning_rate):
    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}

    for epoch in range(epoch_num):
        perm = np.random.permutation(train_size)
        x_train_shuffled = x_train[perm]
        y_train_shuffled = y_train[perm]
        epoch_losses = []

        for i in range(0, train_size, batch_size):
            x_batch = x_train_shuffled[i:i + batch_size]
            y_batch = y_train_shuffled[i:i + batch_size]

            grads = net.gradient(x_batch, y_batch)
            epoch_losses.append(net.last_layer.loss)

            for key in net.params.keys():
                net.params[key] -= learning_rate * grads[key]

        train_loss = float(np.mean(epoch_losses))
        train_acc = net.accuracy(x_train, y_train)
        test_acc = net.accuracy(x_test, y_test)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)

        print(f'Epoch {epoch + 1}/{epoch_num}, Loss: {train_loss:.4f}, ' +
              f'Train Accuracy: {train_acc:.4f}, Test Accuracy: {test_acc:.4f}')

    return history


## Start training...

In [ ]:
history = train(net, learning_rate)

In [ ]:
import pickle

with open('Lee_mnist_model.pkl', 'wb') as f:
    pickle.dump(net.params, f)

print('Saved Lee_mnist_model.pkl')


## Test the Trained Net with Test Images

In [ ]:
# test the trained model on the test set
test_accuracy = net.accuracy(x_test, y_test)
print(f"Final Test Accuracy: {test_accuracy:.4f}")

In [ ]:
# Test the trained model with a single image from the test set
import matplotlib.pyplot as plt

# randomly select an index from the test set
index = np.random.randint(0, x_test.shape[0])
x_single = x_test[index].reshape(1, -1)  # reshape to (1, 784)
y_single = y_test[index].reshape(1, -1)  # reshape to (1, 10)
y_pred_single = net.predict(x_single)
predicted_label = np.argmax(y_pred_single)
true_label = np.argmax(y_single)
print(f"Predicted Label: {predicted_label}, True Label: {true_label}")

# visualize the image and the prediction
plt.imshow(x_single.reshape(28, 28), cmap='gray')
plt.title(f"Predicted: {predicted_label}, True: {true_label}")
plt.axis('off')
plt.show()  


## Test with My Handwritten Digits
Place the 50 images in the `images` folder using names such as `0_1.png` to `9_5.png`.

In [ ]:
from module07 import test_handwritten_images

test_handwritten_images(net)
